<a href="https://colab.research.google.com/github/Vishal-Kawade00/Agentic-Ai/blob/main/ai-engine/Hybrid_RAG_Inference_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CELL 1: DEPENDENCIES & ENVIRONMENT SET UP
# ==========================================
!pip install -q -U fastapi uvicorn pyngrok python-multipart
!pip install -q -U pymupdf chromadb langchain-huggingface google-genai
# CRITICAL: These two provide the SemanticChunker and the HF integration
!pip install -q -U langchain-experimental langchain-community

print("✅ Enterprise AI dependencies installed successfully.")

In [ ]:
# ==========================================
# CELL 3: OPTIMIZED LOCAL INGESTION (REFINED)
# ==========================================
import fitz  # PyMuPDF
import uuid
import chromadb
import os
import logging
from chromadb.utils.embedding_functions import EmbeddingFunction
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

# Mute standard HF warnings to keep the logs clean as per your requirement
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# 1. OPTIMIZED INITIALIZATION
# ==========================================
print("[SYSTEM] Initializing local semantic models...")

# Load the HuggingFace model once.
# NOTE: If you have an HF_TOKEN, add it to Colab secrets as 'HF_TOKEN'
# to avoid rate limit warnings, though it's optional for this model.
hf_embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
semantic_chunker = SemanticChunker(hf_embedder, breakpoint_threshold_type="percentile")

# Fixed the DeprecationWarning by adding __init__
class HFWrapper(EmbeddingFunction):
    def __init__(self, embedder):
        self.embedder = embedder

    def __call__(self, input_texts):
        return self.embedder.embed_documents(input_texts)

db_path = "./chroma_db"
if not os.path.exists(db_path):
    os.makedirs(db_path)

chroma_client = chromadb.PersistentClient(path=db_path)

# Pass the instance of hf_embedder to the wrapper
embedding_fn = HFWrapper(hf_embedder)

vector_db = chroma_client.get_or_create_collection(
    name="rag_documents",
    embedding_function=embedding_fn
)

# ==========================================
# 2. LAYOUT-AWARE PARSER
# ==========================================
def parse_pdf_layout(pdf_bytes):
    """Fast layout-aware parsing for multi-column documents."""
    doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    if len(doc) == 0:
        raise ValueError("Empty or corrupt PDF file.")

    text_data = []
    for page in doc:
        page_width = page.rect.width
        mid = page_width / 2
        blocks = page.get_text("blocks")

        # Sort left column then right column
        left = sorted([b for b in blocks if b[0] < mid], key=lambda b: b[1])
        right = sorted([b for b in blocks if b[0] >= mid], key=lambda b: b[1])

        page_text = "\n".join(b[4].replace('\n', ' ').strip() for b in (left + right) if b[6] == 0)
        if page_text:
            text_data.append(page_text)
    return text_data

# ==========================================
# 3. THE MASTER ORCHESTRATOR
# ==========================================
def process_and_store_semantic(pdf_bytes, filename):
    """Ingests PDF locally with semantic precision."""
    # 1. Parse Layout
    text_data = parse_pdf_layout(pdf_bytes)

    # 2. Semantic Chunking
    full_document_text = "\n\n".join(text_data)
    text_chunks = semantic_chunker.split_text(full_document_text)

    documents, metadatas, ids = [], [], []

    # 3. Batch Preparation
    for chunk in text_chunks:
        documents.append(chunk)
        metadatas.append({
            "source": filename,
            "type": "text",
            "page_number": "multi"
        })
        ids.append(f"{filename}_{uuid.uuid4().hex[:8]}")

    # 4. Vector Storage
    if documents:
        vector_db.add(documents=documents, metadatas=metadatas, ids=ids)

    return len(documents), {"title": filename, "document_type": "PDF"}

In [ ]:
# ==========================================
# CELL 4: OPTIMIZED ASYNC INGESTION ENDPOINT
# ==========================================
from fastapi import FastAPI, UploadFile, File, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from contextlib import asynccontextmanager
from pydantic import BaseModel
from typing import Optional, Dict, Any
import time, uuid, asyncio, threading
from functools import partial

# --- GLOBAL STATE & LOCKS ---
ingestion_jobs = {}
jobs_lock = threading.Lock()
ingestion_lock = threading.Lock()
MAX_FILE_SIZE = 50 * 1024 * 1024

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup: You can initialize your ChromaDB client here if preferred
    yield
    # Shutdown: Clean up resources

app = FastAPI(title="Optimized RAG Engine", lifespan=lifespan)

# Allow all origins for seamless Node.js Gateway proxying
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class JobStatusResponse(BaseModel):
    status: str
    chunks: Optional[int] = None
    time_sec: Optional[float] = None
    error: Optional[str] = None
    metadata: Optional[Dict[str, Any]] = None
    created_at: float

def cleanup_old_jobs():
    """Silently prevents memory leaks by purging old job records."""
    cutoff = time.time() - 3600
    with jobs_lock:
        to_delete = [jid for jid, j in ingestion_jobs.items() if j.get("created_at", 0) < cutoff]
        for jid in to_delete:
            del ingestion_jobs[jid]

def background_ingestion_task(job_id: str, file_bytes: bytes, filename: str):
    """Executes the Cell 3 pipeline locally off the main event loop."""
    if not ingestion_lock.acquire(blocking=False):
        with jobs_lock:
            ingestion_jobs[job_id].update({
                "status": "failed",
                "error": "System busy. Another document is currently being indexed."
            })
        return

    try:
        with jobs_lock:
            ingestion_jobs[job_id]["status"] = "processing"

        start_time = time.time()

        # Executes the optimized local ingestion from Cell 3
        total_chunks, doc_metadata = process_and_store_semantic(file_bytes, filename)

        with jobs_lock:
            ingestion_jobs[job_id].update({
                "status": "completed",
                "metadata": doc_metadata,
                "chunks": total_chunks,
                "time_sec": round(time.time() - start_time, 2)
            })

    except Exception as e:
        with jobs_lock:
            ingestion_jobs[job_id].update({"status": "failed", "error": str(e)})
    finally:
        with jobs_lock:
            if ingestion_jobs.get(job_id, {}).get("status") == "processing":
                ingestion_jobs[job_id].update({"status": "failed", "error": "Internal execution error."})
        ingestion_lock.release()

# ==========================================
# ENDPOINTS
# ==========================================
@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/process-pdf")
async def process_pdf(background_tasks: BackgroundTasks, file: UploadFile = File(...)):
    """Accepts PDF and initiates background processing."""
    contents = await file.read()

    if len(contents) > MAX_FILE_SIZE:
        raise HTTPException(status_code=413, detail="File exceeds 50MB limit.")
    if not contents[:4] == b'%PDF':
        raise HTTPException(status_code=400, detail="Invalid file format. Please upload a PDF.")

    job_id = str(uuid.uuid4())
    with jobs_lock:
        ingestion_jobs[job_id] = {"status": "queued", "created_at": time.time()}

    # Hand off the task to the executor
    loop = asyncio.get_event_loop()
    background_tasks.add_task(
        loop.run_in_executor,
        None,
        partial(background_ingestion_task, job_id, contents, file.filename)
    )

    return {"status": "accepted", "job_id": job_id}

@app.get("/ingest-status/{job_id}", response_model=JobStatusResponse)
def get_ingest_status(job_id: str):
    """Polling endpoint for the frontend to track real-time progress."""
    cleanup_old_jobs()
    with jobs_lock:
        job = ingestion_jobs.get(job_id)

    if not job:
        raise HTTPException(status_code=404, detail="Job ID not found or session expired.")
    return job

In [ ]:
# ==========================================
# CELL 5: OPTIMIZED ROUTING & GENERATION (SDK v2)
# ==========================================
from pydantic import BaseModel
from typing import List, Dict, Optional
from fastapi import HTTPException
from fastapi.responses import StreamingResponse
import json
import asyncio
from google import genai
from google.genai import types
from google.colab import userdata

# --- 1. MODEL DEFINITIONS ---
# Define these BEFORE the routes to avoid NameErrors
class QueryRequest(BaseModel):
    query: str
    chat_history: Optional[List[Dict[str, str]]] = []

# --- CACHE & GLOBALS ---
query_cache = {}
MAX_CACHE_SIZE = 50
MODEL_ID = "gemini-2.5-flash"

# --- 🚨 GLOBAL SAFETY OVERRIDE 🚨 ---
GLOBAL_SAFETY = [
    types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
    types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
]

# Initialize Client safely
try:
    client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
except Exception as e:
    print(f"❌ [AUTH ERROR] Gemini Client failed: {e}")

# ==========================================
# 2. FAST ROUTING & GENERATION UTILS
# ==========================================
async def determine_intent(query: str) -> str:
    """Classifies query using a fast, cheap prompt."""
    prompt = f'Classify as "SPECIFIC" or "GLOBAL". Return EXACTLY ONE WORD.\nQuery: "{query}"'
    try:
        resp = await asyncio.to_thread(
            client.models.generate_content,
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(safety_settings=GLOBAL_SAFETY)
        )
        return "GLOBAL" if "GLOBAL" in resp.text.upper() else "SPECIFIC"
    except Exception as e:
        print(f"⚠️ [ROUTER WARN] Intent check failed: {e}")
        return "SPECIFIC"

async def specific_rag_stream(query: str, contexts: List[str], metadata_packet: dict):
    """Streams the final answer natively using SDK v2."""
    yield json.dumps({"type": "meta", "data": metadata_packet}) + "\n"

    context_str = "\n---\n".join(contexts)
    prompt = f"Answer STRICTLY using context.\nCONTEXT:\n{context_str}\n\nQUESTION:\n{query}"

    try:
        stream = await asyncio.to_thread(
            client.models.generate_content_stream,
            model=MODEL_ID,
            contents=prompt,
            config=types.GenerateContentConfig(safety_settings=GLOBAL_SAFETY)
        )

        for chunk in stream:
            if chunk.text:
                yield json.dumps({"type": "text", "text": chunk.text}) + "\n"

    except Exception as e:
        error_msg = f"\n[API Error: {str(e)}]"
        yield json.dumps({"type": "text", "text": error_msg}) + "\n"
        print(f"❌ [GENERATION ERROR] {e}")

# ==========================================
# 3. THE MASTER ENDPOINT (/ask)
# ==========================================
# Note: Ensure 'app' is the FastAPI instance defined in Cell 4
@app.post("/ask")
async def ask_question(request: QueryRequest):
    try:
        print(f"💬 [CHAT] Query: '{request.query}'")

        standalone_query = request.query
        if request.chat_history:
            history_str = "\n".join([f"{m['role']}: {m['content']}" for m in request.chat_history[-3:]])
            prompt = f"Rewrite to standalone query.\nHISTORY: {history_str}\nQUERY: {standalone_query}"
            try:
                resp = await asyncio.to_thread(
                    client.models.generate_content,
                    model=MODEL_ID,
                    contents=prompt,
                    config=types.GenerateContentConfig(safety_settings=GLOBAL_SAFETY)
                )
                if resp.text:
                    standalone_query = resp.text.strip()
            except Exception as e:
                print(f"⚠️ [MEMORY WARN] Memory rewrite failed: {e}")

        # Cache/Retrieval logic
        contexts = query_cache.get(standalone_query)
        intent = "SPECIFIC"

        if not contexts:
            intent = await determine_intent(standalone_query)
            if intent == "GLOBAL":
                # Ensure vector_db is initialized in Cell 3
                results = vector_db.get()
                contexts = results['documents'][:30] if results['documents'] else []
            else:
                results = vector_db.query(query_texts=[standalone_query], n_results=5)
                contexts = results['documents'][0] if results['documents'] else []

            if contexts:
                query_cache[standalone_query] = contexts
                if len(query_cache) >= MAX_CACHE_SIZE:
                    query_cache.pop(next(iter(query_cache)))

        metadata_packet = {
            "search_type": "Agentic Summary" if intent == "GLOBAL" else "Semantic Search",
            "citations": [{"page": "multi", "text": (ctx[:120] + "...")} for ctx in contexts]
        }

        if not contexts:
            async def no_ctx():
                yield json.dumps({"type": "meta", "data": metadata_packet}) + "\n"
                yield json.dumps({"type": "text", "text": "Please index a PDF first."}) + "\n"
            return StreamingResponse(no_ctx(), media_type="application/x-ndjson")

        return StreamingResponse(
            specific_rag_stream(standalone_query, contexts, metadata_packet),
            media_type="application/x-ndjson"
        )

    except Exception as e:
        print(f"❌ [API ERROR] {str(e)}")
        raise HTTPException(status_code=500, detail="Internal AI Engine Error")

In [ ]:
# ==========================================
# FINAL CELL: THE SECURE & RESILIENT SERVER
# ==========================================
from pyngrok import ngrok
import uvicorn
from google.colab import userdata
import asyncio
import socket
import logging
import urllib.request
import atexit

# --- 1. SILENT LOGGING ---
logging.basicConfig(
    filename="rag_server.log",
    level=logging.WARNING,
    format="%(asctime)s %(levelname)s %(message)s"
)
logging.getLogger("uvicorn.access").setLevel(logging.ERROR)

# --- 2. PORT CLEANUP ---
port = int(8000)
print(f"🧹 [1/4] Cleaning port {port}...")
!fuser -k 8000/tcp

# --- 3. SECURE TUNNEL INITIALIZATION ---
print(f"🔑 [2/4] Initializing Ngrok Auth...")
try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Close orphaned tunnels
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
except Exception as e:
    print(f"⚠️ Ngrok Setup Note: {e}")

# --- 4. RESILIENT HEALTH CHECK ---
async def wait_for_server(public_url):
    """Polls health endpoint with a longer timeout (40 seconds)."""
    for i in range(40):
        try:
            # Pinging the local interface
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2)
            print(f"\n🌐 [READY] RAG Engine is Live")
            print(f"🔗 Tunnel URL: {public_url}")
            print(f"🚀 Update your Node.js .env with this URL!\n")
            return
        except Exception:
            await asyncio.sleep(1)

    print("⚠️ Health check timed out. Check rag_server.log for FastAPI errors.")

# --- 5. THE STARTUP ORCHESTRATOR ---
async def start_server():
    print(f"🚀 [3/4] Starting FastAPI Server on port {port}...")
    config = uvicorn.Config(
        app,
        host="127.0.0.1",
        port=port,
        loop="asyncio",
        workers=1,
        log_level="error",
        access_log=False
    )
    server = uvicorn.Server(config)

    # Start server task
    server_task = asyncio.create_task(server.serve())

    # Give Uvicorn 3 seconds to bind to the port before opening Ngrok
    await asyncio.sleep(3)

    print(f"📡 [4/4] Opening Ngrok Tunnel...")
    try:
        public_url = ngrok.connect(port).public_url
        asyncio.create_task(wait_for_server(public_url))
        await server_task
    except (asyncio.CancelledError, KeyboardInterrupt):
        pass
    finally:
        ngrok.kill()
        print("\n✅ Server and tunnel shut down cleanly.")

# Ignite the Engine
await start_server()

In [ ]:
# ==========================================
# CELL: THE EVALUATION HARNESS
# ==========================================
import json

# 1. Define the Ground Truth (The Test Answers)
test_cases = [
    {
        "query": "What is a Primary Key?",
        "expected_keywords": ["unique", "identifier", "record", "null"]
    },
    {
        "query": "What does SQL stand for?",
        "expected_keywords": ["structured", "query", "language"]
    },
    {
        "query": "Summarize the entire document.",
        "expected_keywords": ["database", "management", "system"] # Checking if GLOBAL route fires
    },
    {
        "query": "What are the advantages of it?", # Testing conversational memory
        "chat_history": [{"role": "user", "content": "What is a relational database?"}, {"role": "ai", "content": "A relational database uses tables."}],
        "expected_keywords": ["tables", "data", "integrity", "relationships"]
    }
]

def run_evaluation():
    print("🧪 Starting Baseline Evaluation...\n")
    score = 0
    total = len(test_cases)

    for i, test in enumerate(test_cases):
        print(f"[{i+1}/{total}] Testing: '{test['query']}'")

        # Determine intent & memory (simulating our API route)
        history = test.get("chat_history", [])
        standalone_query = reformulate_query(test["query"], history) if history else test["query"]
        intent = determine_intent(standalone_query)

        # Generate Answer
        if intent == "GLOBAL":
            answer = generate_global_summary(standalone_query)
        else:
            contexts, citations = retrieve_and_rerank(standalone_query)
            answer = generate_grounded_answer(standalone_query, contexts) if contexts else ""

        answer_lower = answer.lower()

        # Check if the AI hit the expected keywords
        matches = [kw for kw in test['expected_keywords'] if kw.lower() in answer_lower]

        if len(matches) > 0:
            print(f"  ✅ PASS: Found keywords {matches}")
            score += 1
        else:
            print(f"  ❌ FAIL: Missing expected keywords {test['expected_keywords']}")
            print(f"     AI Answer: {answer[:100]}...\n")

    print("-" * 30)
    print(f"🏆 Final Recall Score: {score}/{total} ({(score/total)*100:.1f}%)")

# Run the test!
run_evaluation()